# Notebook 03 — DML Delta Lake (INSERT · UPDATE · DELETE)

Demonstra as operações de DML nas tabelas Delta do bucket `bronze`, além de consultar o histórico e versões anteriores.

| Operação | Tabela alvo | O que faz |
|----------|-------------|----------|
| INSERT | `credor` | Adiciona 3 novos fornecedores via MERGE |
| INSERT | `empenho` | Adiciona novo empenho via SQL |
| UPDATE | `empenho` | Empenhos `Anulado` → `Cancelado` |
| UPDATE | `credor` | Normaliza municipio dos credores de SP |
| DELETE | `pagamento` | Remove pagamentos com situacao `Cancelado` |
| HISTORY | todas | Exibe o log de versoes de cada tabela |
| TIME TRAVEL | `empenho`, `credor` | Lê versão 0 (antes de qualquer DML) |

In [1]:
import os
from dotenv import load_dotenv
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable

load_dotenv()

MINIO_ENDPOINT   = os.getenv('MINIO_ENDPOINT',   'http://localhost:9020')
MINIO_ACCESS_KEY = os.getenv('MINIO_ACCESS_KEY', 'minioadmin')
MINIO_SECRET_KEY = os.getenv('MINIO_SECRET_KEY', 'minioadmin')
BUCKET_BRONZE    = os.getenv('MINIO_BUCKET_BRONZE', 'bronze')

def bronze(tabela): return f's3a://{BUCKET_BRONZE}/{tabela}'

print(f'Bronze: s3a://{BUCKET_BRONZE}')


Bronze: s3a://bronze


## 0. SparkSession

In [2]:
# 1. Defina os pacotes extras em uma lista (sem incluir o Delta aqui)
EXTRA_PACKAGES = [
    "org.apache.hadoop:hadoop-aws:3.3.4",
    "com.amazonaws:aws-java-sdk-bundle:1.12.367"
]

# 2. Crie o builder sem a linha do config('spark.jars.packages')
builder = (
    SparkSession.builder
    .appName('DespesaPublica-03-DML-Delta')
    .config('spark.sql.extensions',
            'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog',
            'org.apache.spark.sql.delta.catalog.DeltaCatalog')
    .config('spark.hadoop.fs.s3a.endpoint',          MINIO_ENDPOINT)
    .config('spark.hadoop.fs.s3a.access.key',        MINIO_ACCESS_KEY)
    .config('spark.hadoop.fs.s3a.secret.key',        MINIO_SECRET_KEY)
    .config('spark.hadoop.fs.s3a.path.style.access', 'true')
    .config('spark.hadoop.fs.s3a.impl',
            'org.apache.hadoop.fs.s3a.S3AFileSystem')
    .config('spark.hadoop.fs.s3a.connection.ssl.enabled', 'false') # Adicionado preventivamente
)

# 3. Injete os pacotes nativamente pelo utilitário do Delta
spark = configure_spark_with_delta_pip(builder, extra_packages=EXTRA_PACKAGES).getOrCreate()

spark.sparkContext.setLogLevel('WARN')

# Re-registrar tabelas no catalogo (necessario se kernel foi reiniciado)
TABELAS = [
    'orgao','unidade','programa','acao','fonte_recurso',
    'credor','empenho','item_empenho','liquidacao','pagamento','historico_preco'
]

spark.sql('CREATE DATABASE IF NOT EXISTS despesa_publica')
for t in TABELAS:
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS despesa_publica.{t}
        USING DELTA LOCATION '{bronze(t)}'
    """)

print(f'Spark {spark.version} | {len(TABELAS)} tabelas registradas')

26/05/05 13:51:25 WARN Utils: Your hostname, DESKTOP-6KNCQNJ resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/05/05 13:51:25 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/mendax/spark-delta-minio-despesa/spark-delta-minio-despesa/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/mendax/.ivy2/cache
The jars for the packages stored in: /home/mendax/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-7efe4b0a-0d4f-4eb2-90f2-b8489dd45c7b;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.367 in central
:: resolution report :: resolve 735ms :: artifacts dl 71ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.367 from central in [default]
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-

Spark 3.5.3 | 11 tabelas registradas


## 1. Estado inicial das tabelas que sofrerão DML

In [3]:
print('--- CREDOR ---')
spark.sql('SELECT COUNT(*) AS total FROM despesa_publica.credor').show()
spark.sql(
    'SELECT id_credor, razao_social, municipio, uf '
    'FROM despesa_publica.credor ORDER BY id_credor DESC LIMIT 5'
).show(truncate=40)

print('--- EMPENHO por situacao ---')
spark.sql(
    'SELECT situacao, COUNT(*) AS qtd, ROUND(SUM(valor_empenho),2) AS valor_total '
    'FROM despesa_publica.empenho GROUP BY situacao ORDER BY situacao'
).show()

print('--- PAGAMENTO ---')
spark.sql(
    'SELECT situacao, COUNT(*) AS qtd '
    'FROM despesa_publica.pagamento GROUP BY situacao'
).show()


--- CREDOR ---


26/05/05 13:52:07 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

+-----+
|total|
+-----+
|   15|
+-----+



+---------+----------------------+-------------+---+
|id_credor|          razao_social|    municipio| uf|
+---------+----------------------+-------------+---+
|       15|       Limpeza Nu Ltda|        Natal| RN|
|       14|       Seguranca Mu SA|     Sao Luis| MA|
|       13|Manutencao Lambda Ltda|Florianopolis| SC|
|       12|     Software Kappa SA|        Belem| PA|
|       11| Construcoes Iota Ltda|      Goiania| GO|
+---------+----------------------+-------------+---+

--- EMPENHO por situacao ---


+---------+---+-------------+
| situacao|qtd|  valor_total|
+---------+---+-------------+
|  Anulado| 14|1.099086463E7|
|   Normal| 19|1.709853109E7|
|Reforcado| 17|1.903670989E7|
+---------+---+-------------+

--- PAGAMENTO ---


+--------+---+
|situacao|qtd|
+--------+---+
|    Pago| 35|
+--------+---+



## 2. INSERT — Novos credores (MERGE)

> Usamos `merge` para garantir idempotência: se o credor já existir não duplica.

In [4]:
dt_credor = DeltaTable.forPath(spark, bronze('credor'))

novos = spark.createDataFrame([
    (16, '99.888.777/0001-66', 'Inovacao Digital Ltda', 'Recife',     'PE', 1),
    (17, '88.777.666/0001-55', 'AgroTech Brasil SA',    'Uberlandia', 'MG', 1),
    (18, '77.666.555/0001-44', 'Energia Verde Ltda',    'Cuiaba',     'MT', 1),
], ['id_credor','cnpj','razao_social','municipio','uf','ativo'])

novos = (
    novos
    .withColumn('dt_carga', F.current_timestamp())
    .withColumn('origem',   F.lit('sqlserver -> landing-zone -> bronze'))
)

(
    dt_credor.alias('dst')
    .merge(novos.alias('src'), 'dst.id_credor = src.id_credor')
    .whenNotMatchedInsertAll()
    .execute()
)

print('3 credores inseridos (MERGE)!')
spark.sql(
    'SELECT id_credor, razao_social, municipio, uf '
    'FROM despesa_publica.credor ORDER BY id_credor DESC LIMIT 5'
).show(truncate=40)


3 credores inseridos (MERGE)!


[Stage 61:>                                                         (0 + 4) / 4]

+---------+---------------------+----------+---+
|id_credor|         razao_social| municipio| uf|
+---------+---------------------+----------+---+
|       18|   Energia Verde Ltda|    Cuiaba| MT|
|       17|   AgroTech Brasil SA|Uberlandia| MG|
|       16|Inovacao Digital Ltda|    Recife| PE|
|       15|      Limpeza Nu Ltda|     Natal| RN|
|       14|      Seguranca Mu SA|  Sao Luis| MA|
+---------+---------------------+----------+---+



## 3. INSERT — Novo empenho via Spark SQL

In [5]:
spark.sql("""
    INSERT INTO despesa_publica.empenho
    SELECT
        51                                    AS id_empenho,
        '2024NE000051'                        AS numero_empenho,
        3                                     AS id_unidade,
        5                                     AS id_acao,
        2                                     AS id_fonte,
        16                                    AS id_credor,
        DATE('2024-12-01')                    AS data_empenho,
        850000.00                             AS valor_empenho,
        'Ordinario'                           AS tipo_empenho,
        'Pregao Eletronico'                   AS modalidade_licitacao,
        'Normal'                              AS situacao,
        'Servicos de TI para o INEP'          AS descricao,
        current_timestamp()                   AS dt_carga,
        'sqlserver -> landing-zone -> bronze' AS origem
""")

print('Empenho 51 inserido!')
spark.sql(
    'SELECT id_empenho, numero_empenho, valor_empenho, situacao '
    'FROM despesa_publica.empenho WHERE id_empenho >= 49'
).show()


Empenho 51 inserido!


+----------+--------------+-------------+---------+
|id_empenho|numero_empenho|valor_empenho| situacao|
+----------+--------------+-------------+---------+
|        49|  2024NE000049|   1355239.84|Reforcado|
|        50|  2024NE000050|   1238108.59|Reforcado|
|        51|  2024NE000051|     850000.0|   Normal|
+----------+--------------+-------------+---------+



## 4. UPDATE — Empenhos `Anulado` → `Cancelado`

In [6]:
print('Antes do UPDATE:')
spark.sql(
    "SELECT id_empenho, numero_empenho, situacao "
    "FROM despesa_publica.empenho WHERE situacao = 'Anulado'"
).show()

dt_empenho = DeltaTable.forPath(spark, bronze('empenho'))
dt_empenho.update(
    condition = F.col('situacao') == 'Anulado',
    set = {
        'situacao':  F.lit('Cancelado'),
        'descricao': F.concat(F.col('descricao'), F.lit(' [CANCELADO 2024-12-31]')),
        'dt_carga':  F.current_timestamp()
    }
)

print('\nApos UPDATE — situacao Cancelado:')
spark.sql(
    "SELECT id_empenho, numero_empenho, situacao "
    "FROM despesa_publica.empenho WHERE situacao = 'Cancelado'"
).show()


Antes do UPDATE:
+----------+--------------+--------+
|id_empenho|numero_empenho|situacao|
+----------+--------------+--------+
|         2|  2024NE000002| Anulado|
|         4|  2024NE000004| Anulado|
|         9|  2024NE000009| Anulado|
|        10|  2024NE000010| Anulado|
|        13|  2024NE000013| Anulado|
|        14|  2024NE000014| Anulado|
|        19|  2024NE000019| Anulado|
|        20|  2024NE000020| Anulado|
|        21|  2024NE000021| Anulado|
|        22|  2024NE000022| Anulado|
|        32|  2024NE000032| Anulado|
|        34|  2024NE000034| Anulado|
|        40|  2024NE000040| Anulado|
|        45|  2024NE000045| Anulado|
+----------+--------------+--------+




Apos UPDATE — situacao Cancelado:


+----------+--------------+---------+
|id_empenho|numero_empenho| situacao|
+----------+--------------+---------+
|         2|  2024NE000002|Cancelado|
|         4|  2024NE000004|Cancelado|
|         9|  2024NE000009|Cancelado|
|        10|  2024NE000010|Cancelado|
|        13|  2024NE000013|Cancelado|
|        14|  2024NE000014|Cancelado|
|        19|  2024NE000019|Cancelado|
|        20|  2024NE000020|Cancelado|
|        21|  2024NE000021|Cancelado|
|        22|  2024NE000022|Cancelado|
|        32|  2024NE000032|Cancelado|
|        34|  2024NE000034|Cancelado|
|        40|  2024NE000040|Cancelado|
|        45|  2024NE000045|Cancelado|
+----------+--------------+---------+



## 5. UPDATE — Normalizar município dos credores de SP

In [7]:
print('Antes:')
spark.sql(
    "SELECT id_credor, razao_social, municipio, uf "
    "FROM despesa_publica.credor WHERE uf = 'SP'"
).show(truncate=40)

dt_credor = DeltaTable.forPath(spark, bronze('credor'))
dt_credor.update(
    condition = F.col('uf') == 'SP',
    set = {'municipio': F.concat(F.col('municipio'), F.lit(' - Capital'))}
)

print('\nApos UPDATE:')
spark.sql(
    "SELECT id_credor, razao_social, municipio, uf "
    "FROM despesa_publica.credor WHERE uf = 'SP'"
).show(truncate=40)


Antes:
+---------+-----------------+---------+---+
|id_credor|     razao_social|municipio| uf|
+---------+-----------------+---------+---+
|        2|Tech Solutions SA|Sao Paulo| SP|
+---------+-----------------+---------+---+


Apos UPDATE:


[Stage 103:===================================================>   (47 + 3) / 50]

+---------+-----------------+-------------------+---+
|id_credor|     razao_social|          municipio| uf|
+---------+-----------------+-------------------+---+
|        2|Tech Solutions SA|Sao Paulo - Capital| SP|
+---------+-----------------+-------------------+---+



## 6. DELETE — Remover pagamentos cancelados

In [8]:
# Inserir 2 pagamentos cancelados para poder demonstrar o DELETE
spark.sql("""
    INSERT INTO despesa_publica.pagamento
    SELECT 36, 36, 5, '2024OB00000036', DATE('2024-11-15'), 125000.00,
           '001-Banco do Brasil', '1234', 'Cancelado',
           current_timestamp(), 'sqlserver -> landing-zone -> bronze'
    UNION ALL
    SELECT 37, 37, 8, '2024OB00000037', DATE('2024-11-20'),  87500.00,
           '104-Caixa Economica',  '5678', 'Cancelado',
           current_timestamp(), 'sqlserver -> landing-zone -> bronze'
""")

print('Antes do DELETE:')
spark.sql(
    'SELECT situacao, COUNT(*) AS qtd '
    'FROM despesa_publica.pagamento GROUP BY situacao'
).show()

dt_pagamento = DeltaTable.forPath(spark, bronze('pagamento'))
dt_pagamento.delete(condition = F.col('situacao') == 'Cancelado')

print('Apos DELETE (cancelados removidos):')
spark.sql(
    'SELECT situacao, COUNT(*) AS qtd, ROUND(SUM(valor_pago),2) AS total '
    'FROM despesa_publica.pagamento GROUP BY situacao'
).show()


Antes do DELETE:


+---------+---+
| situacao|qtd|
+---------+---+
|     Pago| 35|
|Cancelado|  2|
+---------+---+

Apos DELETE (cancelados removidos):


+--------+---+-------------+
|situacao|qtd|        total|
+--------+---+-------------+
|    Pago| 35|1.888205478E7|
+--------+---+-------------+



## 7. HISTORY — Log de versões das tabelas

In [9]:
for tabela in ['credor', 'empenho', 'pagamento']:
    print(f'\n=== HISTORY: {tabela.upper()} ===')
    DeltaTable.forPath(spark, bronze(tabela)) \
        .history() \
        .select('version', 'timestamp', 'operation', 'operationMetrics') \
        .show(truncate=60)



=== HISTORY: CREDOR ===
+-------+-------------------+---------+------------------------------------------------------------+
|version|          timestamp|operation|                                            operationMetrics|
+-------+-------------------+---------+------------------------------------------------------------+
|      2|2026-05-05 13:52:57|   UPDATE|{numRemovedFiles -> 1, numRemovedBytes -> 3441, numCopied...|
|      1|2026-05-05 13:52:37|    MERGE|{numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, num...|
|      0|2026-05-05 13:44:08|    WRITE|{numFiles -> 1, numOutputRows -> 15, numOutputBytes -> 3441}|
+-------+-------------------+---------+------------------------------------------------------------+


=== HISTORY: EMPENHO ===
+-------+-------------------+---------+------------------------------------------------------------+
|version|          timestamp|operation|                                            operationMetrics|
+-------+-------------------+---------+

## 8. TIME TRAVEL — Consultar versões anteriores

In [10]:
# empenho: comparar versao 0 (original) com a versao atual
print('=== TIME TRAVEL: empenho ===\n')

print('VERSAO 0 (antes de qualquer DML) — situacoes:')
spark.read.format('delta').option('versionAsOf', 0) \
    .load(bronze('empenho')) \
    .groupBy('situacao').count().orderBy('situacao').show()

print('VERSAO ATUAL — situacoes:')
spark.read.format('delta').load(bronze('empenho')) \
    .groupBy('situacao').count().orderBy('situacao').show()


=== TIME TRAVEL: empenho ===

VERSAO 0 (antes de qualquer DML) — situacoes:


+---------+-----+
| situacao|count|
+---------+-----+
|  Anulado|   14|
|   Normal|   19|
|Reforcado|   17|
+---------+-----+

VERSAO ATUAL — situacoes:
+---------+-----+
| situacao|count|
+---------+-----+
|Cancelado|   14|
|   Normal|   20|
|Reforcado|   17|
+---------+-----+



In [ ]:
# credor: comparar total de registros entre versao 0 e atual
print('=== TIME TRAVEL: credor ===\n')

n0  = spark.read.format('delta').option('versionAsOf', 0).load(bronze('credor')).count()
now = spark.read.format('delta').load(bronze('credor')).count()

print(f'Versao 0  : {n0} credores')
print(f'Atual     : {now} credores')
print(f'Inseridos : {now - n0} via MERGE')

print('\nLista na versao 0:')
spark.read.format('delta').option('versionAsOf', 0).load(bronze('credor')) \
    .select('id_credor','razao_social','uf').orderBy('id_credor').show(20, truncate=40)


=== TIME TRAVEL: credor ===



Versao 0  : 15 credores
Atual     : 18 credores
Inseridos : 3 via MERGE

Lista na versao 0:


## 9. Resumo final — todas as tabelas

In [ ]:
print('=' * 60)
print('RESUMO FINAL — DespesaPublicaDB no Bronze (Delta Lake)')
print('=' * 60)
print(f'  {"tabela":<20} {"versao":<8} {"registros"}')
print('  ' + '-' * 38)

for t in TABELAS:
    dt  = DeltaTable.forPath(spark, bronze(t))
    ver = dt.history(1).select('version').first()[0]
    n   = spark.read.format('delta').load(bronze(t)).count()
    print(f'  {t:<20} {ver:<8} {n}')

print('''
TABELAS NAO GERENCIADAS (este projeto)
  Dados   : MinIO s3a://bronze/<tabela>
  Spark   : gerencia apenas metadados
  DROP TABLE : remove metadados, dados permanecem no MinIO

TABELAS GERENCIADAS (laboratorio anterior — DataFrames)
  Dados   : sistema de arquivos gerenciado pelo Spark
  DROP TABLE : remove dados E metadados
''')

spark.stop()
print('Notebook 03 concluido! Pipeline completo.')
